# TUDataset DD Protein Graph Classification with GNNVisualizer

This notebook trains a graph-level GCN on the PyTorch Geometric `TUDataset(name="DD")` benchmark, then renders a 300-500 node protein graph with `GNNVisualizer`.

DD is a bioinformatics graph classification dataset. The TU Dortmund table reports 1,178 graphs, 2 classes, about 284 nodes per graph, and about 716 edges per graph, making it a useful larger-than-OHSU stress test for full-graph GNN visualization.

Source docs: [PyG TUDataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.TUDataset.html) and [TU Dortmund graph datasets](https://chrsmrrs.github.io/datasets/docs/datasets/).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Optional environment variables: `DD_EPOCHS`, `DD_MAX_TRAIN_GRAPHS`, `DD_TARGET_NODES`, `DD_MIN_VIS_NODES`, `DD_MAX_VIS_NODES`, and `DD_HIDDEN_CHANNELS`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("DD_EPOCHS", "6"))
MAX_TRAIN_GRAPHS = int(os.environ.get("DD_MAX_TRAIN_GRAPHS", "160"))
TARGET_NODES = int(os.environ.get("DD_TARGET_NODES", "420"))
MIN_VIS_NODES = int(os.environ.get("DD_MIN_VIS_NODES", "300"))
MAX_VIS_NODES = int(os.environ.get("DD_MAX_VIS_NODES", "500"))
HIDDEN_CHANNELS = int(os.environ.get("DD_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 16


def prepare_graph(data):
    data = data.clone()
    data.x = data.x.float()
    data.y = data.y.view(-1).long()
    return data


def select_graph_near(dataset, target_nodes, min_nodes, max_nodes):
    candidates = [prepare_graph(data) for data in dataset if min_nodes <= int(data.num_nodes) <= max_nodes]
    if not candidates:
        candidates = [prepare_graph(data) for data in dataset]
    return min(candidates, key=lambda data: abs(int(data.num_nodes) - target_nodes))


def split_graphs(graphs, train_fraction=0.8):
    count = len(graphs)
    order = torch.randperm(count).tolist()
    split = max(1, min(count - 1, int(count * train_fraction)))
    train_graphs = [graphs[index] for index in order[:split]]
    test_graphs = [graphs[index] for index in order[split:]]
    return train_graphs, test_graphs


dataset = TUDataset(root=str(repo_root / "data" / "tudataset"), name="DD")
all_graphs = [prepare_graph(dataset[index]) for index in range(min(len(dataset), MAX_TRAIN_GRAPHS))]
train_graphs, test_graphs = split_graphs(all_graphs)
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)
visual_data = select_graph_near(dataset, TARGET_NODES, MIN_VIS_NODES, MAX_VIS_NODES)
query_pair = visual_data.edge_index[:, 0].tolist()

num_features = dataset.num_features
num_classes = dataset.num_classes

display(Markdown(
    f"Loaded **DD** with {len(dataset)} graphs, {num_features} node features, "
    f"and {num_classes} classes. Visual graph: {visual_data.num_nodes} nodes, "
    f"{visual_data.edge_index.size(1)} directed edges."
))

In [ ]:
class DDGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        graph_embedding = global_mean_pool(x, batch)
        return self.classifier(graph_embedding)

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            pred = logits.argmax(dim=1)
            target = batch.y.view(-1).long()
            correct += int((pred == target).sum())
            total += int(target.numel())
    return correct / max(total, 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(logits, batch.y.view(-1).long())
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * batch.num_graphs
        if epoch == 1 or epoch == epochs:
            avg_loss = total_loss / max(len(loader.dataset), 1)
            display(Markdown(f"Epoch {epoch}: train loss {avg_loss:.4f}"))
    return model


model = DDGCN(num_features, HIDDEN_CHANNELS, num_classes)
model = train_model(model, train_loader)
test_acc = accuracy(model, test_loader)
display(Markdown(f"Held-out accuracy on the small demo split: **{test_acc:.3f}**"))

The next cell builds the widget. It keeps the default accelerated renderer path (`renderer="auto"`) and enables a taller auto-fit viewport for the larger graph.

In [ ]:
visualizer = GNNVisualizer(viewportHeight=1120, autoFit=True)
visualizer.add_model(
    data=visual_data,
    model=model.eval(),
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

assert visualizer.renderer == "auto"
assert visualizer.autoFit is True
assert visualizer.viewportHeight == 1120
assert visualizer.modelInfo["conv1"]["type"] == "GCNConv"
assert visualizer.modelInfo["conv1"].get("aggregation") == "gcn-normalized"
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured object | Value |\n"
    "| --- | ---: |\n"
    f"| Nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Edges | {visual_data.edge_index.size(1)} |\n"
    f"| Hidden channels | {HIDDEN_CHANNELS} |\n"
    f"| Viewport height | {visualizer.viewportHeight}px |"
))

In [ ]:
display(visualizer)